# Coffee Standard J25 - fixed-holdout feasibility audit
Menghitung dukungan **identitas sumber independen**, bukan hanya jumlah box. Tidak ada training, inference, atau evaluasi model pada test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, os, shutil, subprocess, sys
from pathlib import Path
REPO=Path('/content/coffee-bean-detection')
BRANCH='codex/coffee-standard-primary-audit'
REMOTE='https://'+'github.com/ediprin/coffee-bean-detection.git'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REMOTE,str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.analysis.coffee_standard_j25_split_feasibility import audit_j25_split_feasibility
from coffee_detector.drive_project import resolve_drive_project_root
REL='datasets/coffee-standard-j25-grouped-v2'
REQUIRED=(f'{REL}/coffee_standard_j25_v2_components.json',f'{REL}/coffee_standard_j25_v2_manifest.json',f'{REL}/coffee_standard_j25_v2_summary.json')
PROJECT=resolve_drive_project_root(required_relative_paths=REQUIRED)
GROUPED=PROJECT/REL
OUTPUT=PROJECT/'evidence/coffee-standard-j25-fixed-holdout-feasibility-v1/coffee_standard_j25_fixed_holdout_feasibility.json'
print('REPO:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GROUPED:',GROUPED)


In [ ]:
result=audit_j25_split_feasibility(
    GROUPED/'coffee_standard_j25_v2_components.json',
    GROUPED/'coffee_standard_j25_v2_manifest.json',
    GROUPED/'coffee_standard_j25_v2_summary.json',
    OUTPUT,
    minimum_heldout_identities=2,
    recommended_heldout_identities=5,
    minimum_heldout_objects=10,
)
print('IDENTITY CEILING:',result['global_theoretical_maximum_common_heldout_identities'])
print('LIMITING CLASSES:',result['limiting_classes'])
print('CURRENT OBJECT FAILURES:',result['current_object_gate_failures'])
print('CURRENT IDENTITY FAILURES:',result['current_identity_gate_failures'])
print('GATES:',result['gates'])
print('DECISION:',result['decision'])
print('NEXT:',result['next_action'])
print('TRAINING AUTHORIZED:',result['training_authorized'])
print('SUMMARY:',result['summary'])


In [ ]:
import pandas as pd
table=pd.DataFrame(result['classes'])[[
    'class_id','class_name','total_objects','total_source_identities',
    'current_minimum_heldout_objects','current_minimum_heldout_identities',
    'theoretical_maximum_common_heldout_identities'
]]
display(table)
print('Kirim tabel dan keputusan ini. Jangan training.')
